# MiniMax M2.7: self-evolving agentic training, by the numbers
Toy reproductions of the post's claims. Benchmark figures below are from the AIMLAPI Oct 2026 review (platform-reported); prices mix MiniMax's official list ($0.30/$1.20) and AIMLAPI's ($0.20/$1.10).

In [ ]:
# Cell 1: benchmark margins (AIMLAPI review set, %)
import numpy as np
bench = {"SWE-Pro": (56.22, 50.0, 54.0), "Terminal Bench 2": (82.4, 74.1, 79.2),
         "Tool-calling acc": (75.8, 72.0, 74.0), "GDPval-AA ELO": (1342, 1298, 1315)}
for name, (m27, opus, gpt5) in bench.items():
    print(f"{name:18s} M2.7={m27:7.2f}  Opus4.6={opus:7.2f}  GPT-5={gpt5:7.2f}  margin vs Opus={m27-opus:+.2f}")
assert all(v[0] > v[1] and v[0] > v[2] for v in bench.values()), "M2.7 must lead this review's set"
# SWE-Pro margin in issues: on 1000 tickets, +6.22pp = 62 extra resolved tickets
print("extra tickets resolved per 1000 vs Opus 4.6:", round((56.22 - 50.0) * 10))

In [ ]:
# Cell 2: cost math — dollars per resolved GitHub issue
# assumptions: ~8K input + 2K output tokens per issue attempt (typical agentic SWE loop)
inp, outp = 8000, 2000
def cost(inp_p, out_p): return inp / 1e6 * inp_p + outp / 1e6 * out_p
m27_official = cost(0.30, 1.20); m27_aiml = cost(0.20, 1.10)
opus = cost(15.00, 75.00); gpt5 = cost(10.00, 30.00)
for label, c in [("M2.7 official", m27_official), ("M2.7 AIMLAPI", m27_aiml), ("Opus 4.6", opus), ("GPT-5", gpt5)]:
    print(f"{label:14s} ${c:8.4f} / issue attempt")
ratio_inp_official = 15.0 / 0.30; ratio_inp_aiml = 15.0 / 0.20
print("input-price ratios vs Opus 4.6:", ratio_inp_official, "x (official),", ratio_inp_aiml, "x (AIMLAPI)")
assert abs(ratio_inp_official - 50.0) < 1e-9 and abs(ratio_inp_aiml - 75.0) < 1e-9
# effective cost per RESOLVED issue = attempt cost / success rate
print("cost per resolved issue: M2.7 official $%.3f vs Opus $%.3f" % (m27_official / 0.5622, opus / 0.50))
assert (m27_official / 0.5622) < (opus / 0.50) / 40, "order-of-magnitude cheaper per resolved issue"

In [ ]:
# Cell 3: toy recursive self-improvement — the OpenClaw-style loop
# score_t+1 = score_t + lr * (1 - score_t/scale) * noise_ok  (diminishing returns + noise)
rng = np.random.default_rng(7)
rounds, score, lr, scale = 101, [0.40], 0.045, 1.0
for _ in range(rounds):
    gain = lr * (1 - score[-1] / scale) * max(0.0, rng.normal(1.0, 0.35))
    score.append(min(score[-1] + gain, 0.62))
score = np.array(score)
first10 = score[10] - score[0]; last10 = score[101] - score[91]
print(f"start={score[0]:.3f}  end={score[101]:.3f}  first-10 gain={first10:.3f}  last-10 gain={last10:.4f}")
assert score[101] > score[0] + 0.10, "recursive rounds must improve"
assert last10 < first10, "gains must plateau (diminishing returns)"
assert score[101] <= 0.62, "ceiling respected"
print("plateau reached: later rounds add <10% of early gains ->", last10 / first10 < 0.1)

In [ ]:
# Cell 4: self-critique vs single-shot fine-tune — error reduction
# model: each self-critique round catches a fraction p of remaining errors, then patches them
rng = np.random.default_rng(3)
p, err = 0.35, 1.0
errors = [err]
for _ in range(12):
    err *= (1 - p * rng.uniform(0.7, 1.3)); errors.append(err)
single_shot = 1.0 * (1 - 0.35)          # one SFT pass catches ~35% once
print(f"after 12 self-critique rounds: residual error {errors[-1]:.3f} vs single-shot {single_shot:.3f}")
assert errors[-1] < single_shot * 0.15, "recursive critique must beat one-shot by ~7x on residuals"
print("residual-error ratio (recursive / one-shot):", round(errors[-1] / single_shot, 3))

In [ ]:
# Cell 5: auto prompt caching in agentic workflows (40-60% claim check)
# 20-step agent loop: 6K-token shared prefix + 0.5K new input + 1K output per step
# dollar cost at official M2.7 prices ($0.30 in / $1.20 out) — outputs are NOT cached
steps, prefix, new_in, new_out, pin, pout = 20, 6000, 500, 1000, 0.30, 1.20
in_nc = steps * (prefix + new_in); out_nc = steps * new_out
in_c = prefix + steps * new_in; out_c = out_nc          # outputs still paid every step
cost_nc = in_nc / 1e6 * pin + out_nc / 1e6 * pout
cost_c = in_c / 1e6 * pin + out_c / 1e6 * pout
saving = 1 - cost_c / cost_nc
print(f"no cache: ${cost_nc:.2f}  vs  auto-cache: ${cost_c:.2f}  ->  saving={saving:.1%}")
assert 0.40 <= saving <= 0.60, "claim band is 40-60%"
# shorter loops save less: the band is workload-dependent (long loops, big prefixes win)
short = (5 * (prefix + new_in)) / 1e6 * pin + (5 * new_out) / 1e6 * pout
short_c = (prefix + 5 * new_in) / 1e6 * pin + (5 * new_out) / 1e6 * pout
print("5-step loop saving:", f"{1 - short_c / short:.1%} (band needs long agent loops)")